In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import timm


/Users/lshahsha/Documents/GitHub/model_tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fine-tuning = starting from a model that already learned useful representations, then training it a little more for your specific task. For example we take DinoV2 and we specifically fine tune it to learn to represent the objects in shepard Metzler as well. Fine-tuning means starting from pretrained weights instead of random weights, and then training the model further for a new task or using new data. OR Fine-tuning = continuing training of a pretrained model under a new objective, new data distribution, or new task constraints.

So we assume we have:
- a pretrained encoder (e.g., DINO, ResNet, or your own encoder trained self-supervised)
- a new task (e.g., CIFAR-10 classification)

Fine-tuning answers:
>How do I adapt a general feature extractor to my dataset and labels?

Practically, there are some new concepts that we need to learn. 

Previously when we were building and training our model, we used nn.Module like nn.Linear to tell pytorch that we have containers containing parameters to be trained. During training with the optimizer.zero_grad() and optimizer.step() we would **update** all the parameters in the containers. During testing, however, we don't want the parameters to be updated and so we used .... 

When we are fine tuning a vision encoder, we don't want the parameters of that encoder to be updated either so we **FREEZE** those parameters. When a parameter, in general, is frozen:
- gradients are not computed for it
- optimizer won’t update it
- **But they are still used in forward passes**



during testing, parameters are effectively not being updated either, so in that sense freezing feels like the same thing that happens at test time. But keep in mind that:
Testing is “everything frozen + no learning + no stochasticity.”
Freezing is only “this part doesn’t learn.”


Fine tuning's two orthogonal axis: 

**Axis A — What stays fixed vs what adapts**

you may only want to update certain blocks of the model!

**Axis B - What objective you train with**

- supervised fine-tuning (classification, regression): the respresentation structure stayes the same but the task may change. Depth Anything is a good example of this. The backbone encoder in this model is Dino (self supervised) but it is fine-tuned with supervision to encode depth information as well. The fine tuning in DepthAnything is with supervision (segmentation and ground truth depth values from LiDAR images)
- self-supervised fine-tuning: this will change the data distribution and inductive biases (NEED TO SAY WHAT INDUCTIVE BIASES ARE). For example you want to fine tune a model trained on ImageNET to also learn to represent Shepard Metzler (polycubes) objects. These novel objects which were not exclusively/directly seen during training will change the representational structure (and data distribution)
- Task structure fine tuning: This will change what the model is *encouraged* to represent
- Domain adaptation fine tuning: This is explained by an example: Natural photos → line drawings

In [ ]:
# A parameter is trainable if requires_grad is True
# param.requires_grad = True

# a parameter is frozen if requires_grad is False
# param.requires_grad = False

Examples where we would freeze a model:

1. Linear probing:
    - fully freeze a model
    - train a new head like a simple linear classifier on top of the frozen model
    - Goal: measure representation quality

2. Full Fine-tune:
    - train the encoder and the new head

3. partial fine-tuning:
    - parts/modules of a model are frozen 
    - unfreeze the last k blocks
    - Goal: adapt without overfitting

#### 

### fine tuning a pre-existing model

